# Customer Feature Modeling

Build two reusable analytics tables from the raw Olist data:

- `order_fact`: one row per delivered order, with item, payment, and review measures.
- `customer_features`: one row per `customer_unique_id`, with RFM-style features.

Child tables are aggregated before they are joined so multiple items, payments, or reviews cannot multiply order revenue. For orders with multiple reviews, the latest review by creation date is retained (answer timestamp breaks ties).

In [1]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Raw data directory not found: {DATA_DIR}")

customers = pd.read_csv(DATA_DIR / "olist_customers_dataset.csv")
orders = pd.read_csv(
    DATA_DIR / "olist_orders_dataset.csv",
    parse_dates=[
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ],
)
order_items = pd.read_csv(DATA_DIR / "olist_order_items_dataset.csv")
payments = pd.read_csv(DATA_DIR / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(
    DATA_DIR / "olist_order_reviews_dataset.csv",
    parse_dates=["review_creation_date", "review_answer_timestamp"],
)

print(f"Raw orders: {len(orders):,}")
print(f"Delivered orders: {orders['order_status'].eq('delivered').sum():,}")

Raw orders: 99,441
Delivered orders: 96,478


## 1. Define the delivered-order population

The first version models delivered orders, matching the completed-purchase population used in the EDA. Keep the order key unique before joining any measures.

In [2]:
delivered_orders = orders.loc[orders["order_status"].eq("delivered")].copy()

assert delivered_orders["order_id"].is_unique
print("Delivered order date range:")
print(delivered_orders["order_purchase_timestamp"].min())
print(delivered_orders["order_purchase_timestamp"].max())

Delivered order date range:
2016-09-15 12:16:38
2018-08-29 15:00:37


## 2. Aggregate child tables to order grain

Order items and payments can have multiple rows per order. Reviews can also have multiple rows; keeping the latest review makes the review measure one-per-order and preserves the order grain.

In [3]:
item_metrics = (
    order_items.groupby("order_id", as_index=False)
    .agg(
        order_item_count=("order_item_id", "count"),
        item_revenue=("price", "sum"),
        freight_value=("freight_value", "sum"),
    )
)

payment_metrics = (
    payments.groupby("order_id", as_index=False)
    .agg(
        payment_row_count=("payment_sequential", "count"),
        payment_value=("payment_value", "sum"),
        payment_method_count=("payment_type", "nunique"),
    )
)

latest_reviews = (
    reviews.sort_values(["review_creation_date", "review_answer_timestamp"])
    .drop_duplicates(subset="order_id", keep="last")
)

assert item_metrics["order_id"].is_unique
assert payment_metrics["order_id"].is_unique
assert latest_reviews["order_id"].is_unique
print(f"Orders with multiple source reviews: {reviews['order_id'].duplicated().sum():,}")

Orders with multiple source reviews: 551


## 3. Build the order-level table

Use left joins for the child-table measures so an order is not silently dropped if a source measure is missing. `validate` checks guard against unexpected join fanout.

In [4]:
customer_attributes = customers[
    ["customer_id", "customer_unique_id", "customer_state"]
]

order_fact = (
    delivered_orders[
        [
            "order_id",
            "customer_id",
            "order_purchase_timestamp",
            "order_delivered_customer_date",
            "order_estimated_delivery_date",
        ]
    ]
    .merge(customer_attributes, on="customer_id", how="left", validate="many_to_one")
    .merge(item_metrics, on="order_id", how="left", validate="one_to_one")
    .merge(payment_metrics, on="order_id", how="left", validate="one_to_one")
    .merge(
        latest_reviews[["order_id", "review_score", "review_creation_date"]],
        on="order_id",
        how="left",
        validate="one_to_one",
    )
)

order_fact["delivery_days"] = (
    order_fact["order_delivered_customer_date"]
    - order_fact["order_purchase_timestamp"]
).dt.total_seconds() / 86400

assert len(order_fact) == len(delivered_orders)
assert order_fact["order_id"].is_unique
assert order_fact["customer_unique_id"].notna().all()

join_coverage = pd.DataFrame(
    {
        "measure": ["items", "payments", "reviews"],
        "orders_present": [
            order_fact["order_item_count"].notna().sum(),
            order_fact["payment_value"].notna().sum(),
            order_fact["review_score"].notna().sum(),
        ],
    }
)
join_coverage["orders_missing"] = len(order_fact) - join_coverage["orders_present"]
display(join_coverage)
display(order_fact.head())

,measure,orders_present,orders_missing
0,items,96478,0
1,payments,96477,1
2,reviews,95832,646


,order_id,customer_id,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_state,order_item_count,item_revenue,freight_value,payment_row_count,payment_value,payment_method_count,review_score,review_creation_date,delivery_days
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,2017-10-02 10:56:33,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,SP,1,29.99,8.72,3.0,38.71,2.0,4.0,2017-10-11,8.436574
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,2018-07-24 20:41:37,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,BA,1,118.70,22.76,1.0,141.46,1.0,4.0,2018-08-08,13.782037
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,2018-08-08 08:38:49,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,GO,1,159.90,19.22,1.0,179.12,1.0,5.0,2018-08-18,9.394213
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,2017-11-18 19:28:06,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,RN,1,45.00,27.20,1.0,72.20,1.0,5.0,2017-12-03,13.208750
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,2018-02-13 21:18:39,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,SP,1,19.90,8.72,1.0,28.62,1.0,5.0,2018-02-17,2.873877


## 4. Build customer-level features

The customer key is `customer_unique_id`, not the order-specific `customer_id`. Recency is measured against the last purchase date in this delivered-order dataset; it is a snapshot feature, not a churn label.

In [5]:
customer_features = (
    order_fact.sort_values("order_purchase_timestamp")
    .groupby("customer_unique_id", as_index=False)
    .agg(
        order_count=("order_id", "nunique"),
        total_item_revenue=("item_revenue", "sum"),
        average_order_value=("item_revenue", "mean"),
        total_freight_value=("freight_value", "sum"),
        first_order_date=("order_purchase_timestamp", "min"),
        last_order_date=("order_purchase_timestamp", "max"),
        customer_state=("customer_state", "last"),
    )
)

analysis_date = order_fact["order_purchase_timestamp"].max().normalize()
customer_features["recency_days"] = (
    analysis_date - customer_features["last_order_date"].dt.normalize()
).dt.days
customer_features["is_repeat_customer"] = customer_features["order_count"].gt(1)

display(customer_features.head())
print("RFM snapshot date:", analysis_date.date())

,customer_unique_id,order_count,total_item_revenue,average_order_value,total_freight_value,first_order_date,last_order_date,customer_state,recency_days,is_repeat_customer
0,0000366f3b9a7992bf8c76cfdf3221e2,1,129.90,129.90,12.00,2018-05-10 10:56:27,2018-05-10 10:56:27,SP,111,False
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,18.90,18.90,8.29,2018-05-07 11:11:27,2018-05-07 11:11:27,SP,114,False
2,0000f46a3911fa3c0805444483337064,1,69.00,69.00,17.22,2017-03-10 21:05:03,2017-03-10 21:05:03,SC,537,False
3,0000f6ccb0745a6a4b88665a16c9f078,1,25.99,25.99,17.63,2017-10-12 20:29:41,2017-10-12 20:29:41,PA,321,False
4,0004aac84e0df4da2b147fca70cf8255,1,180.00,180.00,16.89,2017-11-14 19:45:42,2017-11-14 19:45:42,SP,288,False


RFM snapshot date: 2018-08-29


## 5. Validate and save the tables

Check that the customer table is unique at person grain and reconciles to the delivered-order table before exporting both datasets.

In [6]:
assert customer_features["customer_unique_id"].is_unique
assert customer_features["order_count"].sum() == len(order_fact)
assert customer_features["is_repeat_customer"].equals(
    customer_features["order_count"].gt(1)
)

repeat_rate = customer_features["is_repeat_customer"].mean()
model_summary = pd.DataFrame(
    [
        {
            "delivered_orders": len(order_fact),
            "unique_customers": len(customer_features),
            "repeat_customers": int(customer_features["is_repeat_customer"].sum()),
            "repeat_customer_rate": repeat_rate,
            "item_revenue": order_fact["item_revenue"].sum(),
        }
    ]
)
display(model_summary)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
order_fact.to_csv(OUTPUT_DIR / "order_level.csv", index=False)
customer_features.to_csv(OUTPUT_DIR / "customer_features.csv", index=False)
print(f"Saved tables to {OUTPUT_DIR}")

,delivered_orders,unique_customers,repeat_customers,repeat_customer_rate,item_revenue
0,96478,93358,2801,0.030003,13221498.11


Saved tables to /mnt/c/Users/Nathan Huang/ecommerce-customer-analytics/data/processed
